### tutorial example

In [ ]:
from docetl.api import Pipeline, Dataset, MapOp, UnnestOp, ResolveOp, ReduceOp, PipelineStep, PipelineOutput

# Define the dataset - JSON file with medical transcripts
dataset = Dataset(
    type="file",
    path="./medical_transcripts.json"
)

# Define operations
operations = [
    # Extract medications from each transcript
    MapOp(
        name="extract_medications",
        type="map",
        prompt="""
        Analyze the following transcript of a conversation between a doctor and a patient:
        {{ input.src }}
        Extract and list all medications mentioned in the transcript.
        If no medications are mentioned, return an empty list.
        """,
        output={
            "schema": {
                "medication": "list[str]"
            }
        }
    ),
    # Unnest to create separate items for each medication
    UnnestOp(
        name="unnest_medications",
        type="unnest",
        unnest_key="medication"
    ),
    # Resolve similar medication names
    ResolveOp(
        name="resolve_medications",
        type="resolve",
        blocking_keys=["medication"],
        blocking_threshold=0.6162,
        comparison_prompt="""
        Compare the following two medication entries:
        Entry 1: {{ input1.medication }}
        Entry 2: {{ input2.medication }}
        Are these medications likely to be the same or closely related?
        """,
        embedding_model="text-embedding-3-small",
        output={
            "schema": {
                "medication": "str"
            }
        },
        resolution_prompt="""
        Given the following matched medication entries:
        {% for entry in inputs %}
        Entry {{ loop.index }}: {{ entry.medication }}
        {% endfor %}
        Determine the best resolved medication name for this group of entries. The resolved
        name should be a standardized, widely recognized medication name that best represents
        all matched entries.
        """
    ),
    # Summarize side effects and uses for each medication
    ReduceOp(
        name="summarize_prescriptions",
        type="reduce",
        reduce_key=["medication"],
        prompt="""
        Here are some transcripts of conversations between a doctor and a patient:

        {% for value in inputs %}
        Transcript {{ loop.index }}:
        {{ value.src }}
        {% endfor %}

        For the medication {{ reduce_key }}, please provide the following information based on all the transcripts above:

        1. Side Effects: Summarize all mentioned side effects of {{ reduce_key }}.
        2. Therapeutic Uses: Explain the medical conditions or symptoms for which {{ reduce_key }} was prescribed or recommended.

        Ensure your summary:
        - Is based solely on information from the provided transcripts
        - Focuses only on {{ reduce_key }}, not other medications
        - Includes relevant details from all transcripts
        - Is clear and concise
        - Includes quotes from the transcripts
        """,
        output={
            "schema": {
                "side_effects": "str",
                "uses": "str"
            }
        }
    )
]

# Define pipeline step
step = PipelineStep(
    name="medical_info_extraction",
    input="transcripts",
    operations=[
        "extract_medications",
        "unnest_medications",
        "resolve_medications",
        "summarize_prescriptions"
    ]
)

# Define output
output = PipelineOutput(
    type="file",
    path="./medication_summaries.json",
    intermediate_dir="intermediate_results"
)

# Define system prompt (optional but recommended)
system_prompt = {
    "dataset_description": "a collection of transcripts of doctor visits",
    "persona": "a medical practitioner analyzing patient symptoms and reactions to medications"
}

# Create the pipeline
pipeline = Pipeline(
    name="medical_transcript_analysis",
    datasets={"transcripts": dataset},
    operations=operations,
    steps=[step],
    output=output,
    default_model="gpt-4o-mini",
    system_prompt=system_prompt
)

# Run the pipeline
cost = pipeline.run()
print(f"Pipeline execution completed. Total cost: ${cost:.2f}")

### tutorial example with code synthesis

In [ ]:
from docetl.api import Pipeline, Dataset, MapOp, ReduceOp, PipelineStep, PipelineOutput

# 1. SWAP_WITH_CODE (code substitution) demo pipeline
# Uses medical transcript input and a reduce operation that is ideal for code substitution.
input_path = "./medical_transcripts.json"

swap_dataset = Dataset(type="file", path=input_path)

swap_ops = [
    MapOp(
        name="extract_medications",
        type="map",
        prompt="""
        Extract the medication names from this transcript:
        {{ input.src }}
        Return JSON: {"medication": [str]}
        """,
        output={"schema": {"medication": "list[str]"}},
        optimize=True,
    ),
    ReduceOp(
        name="count_medications",
        type="reduce",
        reduce_key="_all",
        prompt="""
        Compute the total number of medication mentions across all documents and list unique medications.
        Return JSON: {"total_medications": int, "unique_medications": list[str]}
        """,
        output={"schema": {"total_medications": "int", "unique_medications": "list[str]"}},
        optimize=True,
    ),
]

swap_step = PipelineStep(
    name="swap_with_code_step",
    input="transcripts",
    operations=["extract_medications", "count_medications"],
)

swap_output = PipelineOutput(type="file", path="./swap_with_code_output.json")

swap_pipeline = Pipeline(
    name="swap_with_code_demo",
    datasets={"transcripts": swap_dataset},
    operations=swap_ops,
    steps=[swap_step],
    output=swap_output,
    default_model="gpt-4o-mini",
    optimizer_config={
        "type": "moar",
        "save_dir": "./swap_with_code_moar",
        "available_models": ["gpt-4o-mini", "gpt-4o"],
        "evaluation_file": "./example_evaluate.py",
        "metric_key": "precision",
        "max_iterations": 6,
        "rewrite_agent_model": "gpt-4o-mini",
    },
)
import sys

# Redirect output to file
output_file = "./swap_with_code_output.txt"
with open(output_file, "w") as f:
    original_stdout = sys.stdout
    sys.stdout = f
    
    try:
        print("Running swap with code optimization (code substitution directive)")
        optimized_swap_pipeline = swap_pipeline.optimize()
        print("✓ Optimized pipeline generated")
        cost = optimized_swap_pipeline.run()
        print(f"✓ Execution complete. Cost: ${cost:.2f}")
    except Exception as e:
        print(f"Note: {e}")
    finally:
        sys.stdout = original_stdout

print(f"Output saved to {output_file}")

# print("Running swap with code optimization (code substitution directive)")
# try:
#     optimized_swap_pipeline = swap_pipeline.optimize()
#     print("✓ Optimized pipeline generated")
#     cost = optimized_swap_pipeline.run()
#     print(f"✓ Execution complete. Cost: ${cost:.2f}")
# except Exception as e:
#     print(f"Note: {e}")

### deterministic_doc_compression demo

In [ ]:
from docetl.api import Pipeline, Dataset, MapOp, ReduceOp, PipelineStep, PipelineOutput

# 2. Deterministic doc compression demo pipeline
# Input is the same medical transcript dataset; the directive can add a pre-compression step.

compress_dataset = Dataset(type="file", path=input_path)

compress_ops = [
    MapOp(
        name="shorten_transcript",
        type="map",
        prompt="""
        Clean and compress the transcript by removing boilerplate visit script sections and keeping only explicit medication statements:
        {{ input.src }}
        Output schema: {\"compressed_src\": str}
        """,
        output={"schema": {"compressed_src": "str"}},
        optimize=True,
    ),
    MapOp(
        name="extract_medications_after_compression",
        type="map",
        prompt="""
        Extract medications from the compressed content:
        {{ input.compressed_src }}
        Return JSON: {\"medication\": [str]}
        """,
        output={"schema": {"medication": "list[str]"}},
        optimize=True,
    ),
]

compress_step = PipelineStep(
    name="doc_compression_step",
    input="transcripts",
    operations=["shorten_transcript", "extract_medications_after_compression"],
)

compress_output = PipelineOutput(type="file", path="./deterministic_doc_compression_output.json")

compress_pipeline = Pipeline(
    name="deterministic_doc_compression_demo",
    datasets={"transcripts": compress_dataset},
    operations=compress_ops,
    steps=[compress_step],
    output=compress_output,
    default_model="gpt-4o-mini",
    optimizer_config={
        "type": "moar",
        "save_dir": "./deterministic_doc_compression_moar",
        "available_models": ["gpt-4o-mini", "gpt-4o"],
        "evaluation_file": "./example_evaluate.py",
        "metric_key": "precision",
        "max_iterations": 6,
        "rewrite_agent_model": "gpt-4o-mini",
    },
)

print("Running deterministic doc compression optimization")
try:
    optimized_compress_pipeline = compress_pipeline.optimize()
    print("Optimized compression pipeline generated")
    cost = optimized_compress_pipeline.run()
    print(f"Deterministic doc compression run complete, cost: ${cost:.2f}")
except Exception as e:
    print("Deterministic doc compression demo pipeline failed:", e)